# Swiggy Instamart — Delivery Time Regression

**Goal:** Find why average delivery time increased by 4.2 minutes after adding 12 new dark stores.

Download `instamart_orders_q1_2025.csv` from your DayZer0 workspace and place it in `../data/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load & inspect the data

In [ ]:
df = pd.read_csv(
    '../data/instamart_orders_q1_2025.csv',
    parse_dates=['pick_start_ts', 'pack_end_ts', 'dispatch_ts', 'delivered_ts']
)

print(f'Rows: {len(df):,}  |  Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Check for nulls and data quality
df.info()
df.isnull().sum()

## 2. Derive delivery time phases

In [ ]:
df['pick_time_min']     = (df['pack_end_ts']  - df['pick_start_ts']).dt.total_seconds() / 60
df['pack_time_min']     = (df['dispatch_ts']  - df['pack_end_ts']).dt.total_seconds()   / 60
df['lastmile_time_min'] = (df['delivered_ts'] - df['dispatch_ts']).dt.total_seconds()   / 60
df['total_time_min']    = (df['delivered_ts'] - df['pick_start_ts']).dt.total_seconds() / 60

# Sanity check — overall average should be ~22.5 min
df[['pick_time_min', 'pack_time_min', 'lastmile_time_min', 'total_time_min']].describe().round(2)

## 3. New stores vs. existing stores

In [ ]:
phase_cols = ['pick_time_min', 'pack_time_min', 'lastmile_time_min', 'total_time_min']
by_store_type = df.groupby('store_type')[phase_cols].mean().round(2)
print(by_store_type)

by_store_type[['pick_time_min', 'pack_time_min', 'lastmile_time_min']].plot(
    kind='bar', title='Avg time per phase: New vs. Existing stores'
)
plt.ylabel('Minutes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. City-level breakdown

In [ ]:
by_city = (
    df.groupby(['city', 'store_type'])['total_time_min']
    .mean()
    .unstack()
    .round(2)
)
print(by_city)

by_city.plot(kind='bar', title='Avg total delivery time by city')
plt.ylabel('Minutes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Your analysis

Add your findings below. Replace these bullets with your actual conclusions.

**Root cause hypothesis:**
- ...

**Evidence:**
- ...

**Recommendations:**
1. ...
2. ...
3. ...